# Hierarchical emotion classifier

The model reported in *Emotion Classification in Kazakh Social Media: What Limits Performance
in a Low-Resource Setting?* (ICTACS-2026).

A LaBSE encoder with mean pooling and a frozen bottom (embeddings and the lowest four layers)
feeds a shared 256-unit projection and two heads: an eight-way head over the base classes and
a twelve-way head over the affective states, conditioned on the softmax of the first. Training
minimises `0.6 * CE(base) + 0.4 * focal(state)` with balanced class weights. At inference the
score of a state is the sum of its own log-probability and that of its parent.

Input: `dataset_v20.xlsx` (3,073 rows, of which 2,054 authentic) and `gold_v20.xlsx` (413).
The configuration below is the one reported in the paper. The three flags reproduce the
ablation rows of Table V; the evaluation set is never used for model selection.

The notebook also prints a severity grouping of the twelve states, which belongs to a
companion study and is not part of this paper.


In [ ]:
!pip install transformers torch pandas openpyxl scikit-learn -q

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import pandas as pd, numpy as np, random, gc, json
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from torch.amp import autocast, GradScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type=="cuda", "Enable the GPU"
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

# ----- Configuration. The reported model is the setting below. -----
USE_AUGMENTATION = False   # True adds the 1,019 synthetic posts (reported as a negative result)
USE_HIERARCHY    = True    # False gives flat inference, without the parent term
USE_FOCAL        = True    # False replaces the focal loss with a cross-entropy

LABEL_MAP_L1 = {"Joy":0,"Sadness":1,"Anger":2,"Fear":3,"Surprise":4,"Disgust":5,"Neutral":6,"Shame":7}
LABEL_MAP_L2 = {"Joy":0,"Hopelessness":1,"Worthlessness":2,"Exhaustion":3,"Loneliness":4,
                "Frustration":5,"Indignation":6,"Shame":7,"Fear":8,"Surprise":9,
                "Disgust":10,"Neutral":11}
L2_TO_L1 = {"Joy":"Joy","Hopelessness":"Sadness","Worthlessness":"Sadness",
            "Exhaustion":"Sadness","Loneliness":"Sadness","Frustration":"Anger",
            "Indignation":"Anger","Shame":"Shame","Fear":"Fear","Surprise":"Surprise",
            "Disgust":"Disgust","Neutral":"Neutral"}
L2_NAMES=list(LABEL_MAP_L2.keys()); L1_NAMES=list(LABEL_MAP_L1.keys())
L2_PARENT=torch.tensor([LABEL_MAP_L1[L2_TO_L1[n]] for n in L2_NAMES])
# severity grouping used in a companion study: 0 Low, 1 Medium, 2 High
# High = Hopelessness, Worthlessness, Exhaustion | Medium = Loneliness, Frustration,
# Indignation, Fear | Low = the rest
RISK=np.array([0, 2,2,2, 1, 1,1, 0, 1, 0,0,0])
DISTRESS=(RISK>0).astype(int)
NUM_L1, NUM_L2 = 8, 12
SEEDS=[42,123,2026]

In [ ]:
DATA = "/kaggle/input/datasets/tenajuro/datasetv20"
df   = pd.read_excel(f"{DATA}/dataset_v20.xlsx")
gold = pd.read_excel(f"{DATA}/gold_v20.xlsx")

for d in (df, gold):
    d.dropna(subset=["emotion_10","text"], inplace=True)
    assert d["emotion_10"].isin(LABEL_MAP_L2).all(), set(d["emotion_10"])-set(LABEL_MAP_L2)
    d["label_l1"]=d["emotion_10"].map(L2_TO_L1).map(LABEL_MAP_L1)
    d["label_l2"]=d["emotion_10"].map(LABEL_MAP_L2)

assert len(set(df["id"]) & set(gold["id"]))==0, "evaluation ids leaked into the training pool"
assert len(df)==3073 and len(gold)==413, f"wrong files: train {len(df)}, gold {len(gold)}"

real=df[~df.is_augmented].reset_index(drop=True)
aug =df[df.is_augmented].reset_index(drop=True)
y_gold=gold["label_l2"].values
print(f"train {len(df)} (authentic {len(real)}) | evaluation set {len(gold)}")
print(gold["emotion_10"].value_counts().to_string())

In [ ]:
class DS(torch.utils.data.Dataset):
    def __init__(s,t,l1,l2,tok): s.t,s.l1,s.l2,s.tok=t,l1,l2,tok
    def __len__(s): return len(s.t)
    def __getitem__(s,i):
        e=s.tok(str(s.t[i]),max_length=128,padding="max_length",truncation=True,return_tensors="pt")
        return {"input_ids":e["input_ids"].squeeze(),"attention_mask":e["attention_mask"].squeeze(),
                "label_l1":torch.tensor(s.l1[i]),"label_l2":torch.tensor(s.l2[i])}

class FocalLoss(nn.Module):
    def __init__(s,weight=None,gamma=2.0): super().__init__(); s.w,s.g=weight,gamma
    def forward(s,lg,y):
        ce=F.cross_entropy(lg,y,weight=s.w,reduction="none")
        return ((1-torch.exp(-ce))**s.g*ce).mean()

class HierModel(nn.Module):
    def __init__(s,n_frozen=4,dropout=0.3):
        super().__init__()
        s.encoder=AutoModel.from_pretrained("sentence-transformers/LaBSE")
        for p in s.encoder.embeddings.parameters(): p.requires_grad=False
        for n_,p in s.encoder.named_parameters():
            if any(f"layer.{i}." in n_ for i in range(n_frozen)): p.requires_grad=False
        s.shared=nn.Sequential(nn.Linear(768,256),nn.ReLU(),nn.Dropout(dropout))
        s.head_l1=nn.Linear(256,NUM_L1); s.head_l2=nn.Linear(256+NUM_L1,NUM_L2)
    def forward(s,ii,am):
        h=s.encoder(input_ids=ii,attention_mask=am).last_hidden_state
        m=am.unsqueeze(-1).float()
        sh=s.shared((h*m).sum(1)/m.sum(1).clamp(min=1e-9))
        lg1=s.head_l1(sh)
        return lg1, s.head_l2(torch.cat([sh,F.softmax(lg1,-1)],-1))

In [ ]:
def evaluate(model,loader):
    model.eval(); par=L2_PARENT.to(device); P1,Y1,P2,Y2,LOG=[],[],[],[],[]
    with torch.no_grad():
        for b in loader:
            ii,am=b["input_ids"].to(device),b["attention_mask"].to(device)
            with autocast("cuda"): lg1,lg2=model(ii,am)
            lp1=F.log_softmax(lg1.float(),-1); lp2=F.log_softmax(lg2.float(),-1)
            joint = lp2 + lp1[:,par] if USE_HIERARCHY else lp2
            P1.extend(lg1.argmax(1).cpu().numpy()); Y1.extend(b["label_l1"].numpy())
            P2.extend(joint.argmax(1).cpu().numpy()); Y2.extend(b["label_l2"].numpy())
            LOG.append(joint.cpu())
    return (f1_score(Y1,P1,average="macro",zero_division=0),
            f1_score(Y2,P2,average="macro",zero_division=0),
            np.array(P2),np.array(Y2),torch.cat(LOG))

def run(seed,epochs=12,patience=3,bs=16):
    set_seed(seed)
    # Test = the held-out evaluation set; validation is carved from the authentic training posts
    tr_r,val_df=train_test_split(real,test_size=0.15,stratify=real["label_l2"],random_state=seed)
    test_df=gold
    train_df=pd.concat([tr_r,aug]) if USE_AUGMENTATION else tr_r
    tok=AutoTokenizer.from_pretrained("sentence-transformers/LaBSE")
    mk=lambda d: DS(d["text"].tolist(),d["label_l1"].tolist(),d["label_l2"].tolist(),tok)
    trL=torch.utils.data.DataLoader(mk(train_df),batch_size=bs,shuffle=True)
    vaL=torch.utils.data.DataLoader(mk(val_df),batch_size=bs)
    teL=torch.utils.data.DataLoader(mk(test_df),batch_size=bs)
    print(f"  train {len(train_df)} (authentic {len(tr_r)}) | val {len(val_df)} | test {len(test_df)}")
    model=HierModel().to(device)
    w1=torch.tensor(compute_class_weight("balanced",classes=np.arange(NUM_L1),y=train_df["label_l1"]),dtype=torch.float).to(device)
    w2=torch.tensor(compute_class_weight("balanced",classes=np.arange(NUM_L2),y=train_df["label_l2"]),dtype=torch.float).to(device)
    c1 = nn.CrossEntropyLoss(weight=w1)
    c2 = FocalLoss(weight=w2) if USE_FOCAL else nn.CrossEntropyLoss(weight=w2)
    enc=[p for p in model.encoder.parameters() if p.requires_grad]
    heads=list(model.shared.parameters())+list(model.head_l1.parameters())+list(model.head_l2.parameters())
    opt=AdamW([{"params":enc,"lr":2e-5},{"params":heads,"lr":5e-4}],weight_decay=0.01)
    steps=len(trL)*epochs; sch=get_linear_schedule_with_warmup(opt,steps//10,steps); scaler=GradScaler("cuda")
    best,best_state,wait=0,None,0
    for ep in range(epochs):
        model.train()
        for b in trL:
            ii,am=b["input_ids"].to(device),b["attention_mask"].to(device)
            y1,y2=b["label_l1"].to(device),b["label_l2"].to(device)
            opt.zero_grad()
            with autocast("cuda"):
                lg1,lg2=model(ii,am); loss=0.6*c1(lg1,y1)+0.4*c2(lg2,y2)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(opt); scaler.update(); sch.step()
        f1v,f2v,_,_,_=evaluate(model,vaL)
        print(f"  seed {seed} | ep {ep+1:2d} | F1-L1 {f1v:.4f} | F1-L2 {f2v:.4f}")
        if f2v>best: best,wait,best_state=f2v,0,{k:v.cpu().clone() for k,v in model.state_dict().items()}
        else:
            wait+=1
            if wait>=patience: print("  early stop"); break
    model.load_state_dict(best_state)
    t1,t2,preds,labels,logits=evaluate(model,teL)
    print(f"  ✅ seed {seed} GOLD: F1-L1 {t1:.4f} | F1-L2 {t2:.4f}")
    del model; torch.cuda.empty_cache(); gc.collect()
    return t1,t2,preds,labels,logits,best_state

In [ ]:
results,ref=[],None
np.save("/kaggle/working/gold_labels_10.npy", y_gold)
for s in SEEDS:
    f1,f2,preds,labels,logits,state=run(s)
    results.append({"seed":s,"f1_l1":f1,"f1_l2":f2})
    np.save(f"/kaggle/working/probs10_seed{s}.npy", torch.softmax(logits.float(),1).numpy())
    if ref is None or f2>ref["f1_l2"]:
        ref={"labels":labels,"preds":preds,"state":state,"f1_l2":f2,"seed":s}
r=pd.DataFrame(results); print(r.to_string(index=False))
print(f"\nF1-L1: {r.f1_l1.mean():.4f} ± {r.f1_l1.std():.4f}")
print(f"F1-L2: {r.f1_l2.mean():.4f} ± {r.f1_l2.std():.4f}")
torch.save(ref["state"],"/kaggle/working/best_model_10class.pt")

In [ ]:
# ----- Metrics at each granularity: best single seed and the three-seed ensemble -----
PAR=L2_PARENT.numpy()
probs=np.mean([np.load(f"/kaggle/working/probs10_seed{s}.npy") for s in SEEDS],axis=0)
ens=probs.argmax(1)

def ladder(y,p,title):
    print(f"--- {title} ---")
    print(f"12 classes macro-F1: {f1_score(y,p,average='macro',zero_division=0):.4f}")
    print(f" 8 classes macro-F1: {f1_score(PAR[y],PAR[p],average='macro',zero_division=0):.4f}")
    print(f" 3 levels   macro-F1: {f1_score(RISK[y],RISK[p],average='macro',zero_division=0):.4f}")
    print(f" binary     F1: {f1_score(DISTRESS[y],DISTRESS[p],zero_division=0):.4f} "
          f"(P {precision_score(DISTRESS[y],DISTRESS[p],zero_division=0):.3f} / "
          f"R {recall_score(DISTRESS[y],DISTRESS[p],zero_division=0):.3f})")

ladder(y_gold,ref["preds"],f"best single seed {ref['seed']} - evaluation set")
ladder(y_gold,ens,"three-seed ensemble - evaluation set")
print()
print(classification_report(y_gold,ens,labels=range(NUM_L2),target_names=L2_NAMES,zero_division=0,digits=3))
print("=== base emotions ===")
print(classification_report(PAR[y_gold],PAR[ens],labels=range(NUM_L1),target_names=L1_NAMES,zero_division=0,digits=3))
print("=== severity levels ===")
print(classification_report(RISK[y_gold],RISK[ens],labels=[0,1,2],target_names=["Low","Medium","High"],zero_division=0,digits=3))